# Phase D — blinded acoustic review
Listen to neutral-ID WAVs without event/control metadata. Describe only what you hear. An unexplained transient is **not** a germination sound. The mapping is kept separately and should be opened only after all selected clips are labeled. Save each neutral label, confidence and notes with the button.


In [ ]:
from pathlib import Path
import sys
PROJECT=Path.cwd().resolve()
if PROJECT.name=='notebooks': PROJECT=PROJECT.parent
sys.path.insert(0,str(PROJECT))
import pandas as pd
import ipywidgets as widgets
from IPython.display import Audio, display, clear_output
from src.phase_d_audio import save_blind_review
from src.phase_c2_audio_review import ALLOWED_LABELS,ALLOWED_CONFIDENCE
TABLES=PROJECT/'results/tables/targeted_multimodal'
BLIND=TABLES/'phase_d_blind_review.csv'
MANIFEST=TABLES/'phase_d_manual_review_manifest.csv'
assert BLIND.is_file(), 'Run scripts/run_phase_d.py first'
print('Neutral-ID review clips:',len(pd.read_csv(BLIND)))


In [ ]:
table=pd.read_csv(BLIND,keep_default_na=False)
choice=widgets.Dropdown(options=table.blind_id.tolist(),description='Clip:')
label=widgets.Dropdown(options=('',*ALLOWED_LABELS),description='Label:',layout=widgets.Layout(width='520px'))
confidence=widgets.Dropdown(options=('',*ALLOWED_CONFIDENCE),description='Confidence:')
notes=widgets.Textarea(description='Notes:',layout=widgets.Layout(width='700px',height='70px'))
save=widgets.Button(description='Save neutral review',button_style='success')
output=widgets.Output(); status=widgets.Output()
def show(*_):
    current=pd.read_csv(BLIND,keep_default_na=False).loc[lambda d:d.blind_id.eq(choice.value)].iloc[0]
    label.value=current.manual_audio_label if current.manual_audio_label in label.options else ''
    confidence.value=current.manual_audio_confidence if current.manual_audio_confidence in confidence.options else ''
    notes.value=str(current.notes)
    with output:
        clear_output(wait=True)
        print(current.blind_id,'— original-amplitude WAV; source unknown')
        display(Audio(filename=current.listening_path,normalize=False))
def save_row(_):
    with status:
        clear_output(wait=True)
        try:
            save_blind_review(BLIND,MANIFEST,choice.value,label.value,confidence.value,notes.value)
            print('Saved with CSV backups.')
            remaining=pd.read_csv(BLIND,keep_default_na=False).manual_audio_label.eq('').sum()
            print('Unlabeled clips remaining:',remaining)
        except Exception as error:print(type(error).__name__+': '+str(error))
choice.observe(show,names='value');save.on_click(save_row)
display(widgets.VBox([choice,label,confidence,notes,save,status,output]));show()


After all neutral IDs have been reviewed, the separate `results/tables/targeted_multimodal/phase_d_blind_mapping.csv` can be opened to compare event and control labels. This notebook never reads or displays that mapping.
